# VMamba-T from scratch

This experiment constructs the authors' VMamba-T architecture with `pretrained=False`. It loads **no ImageNet or previous experiment checkpoint**, and every backbone and classifier parameter is trained. The exact shared `data/common_split_manifest.csv` provides 176 training and 45 validation images.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.home() / 'Documents' / 'Paper replication'
SHARED = PROJECT_ROOT / 'model_reproductions_7_models' / 'shared'
MODEL_FOLDER = PROJECT_ROOT / 'model_reproductions_7_models' / '04_vmamba_from_scratch'
sys.path.insert(0, str(SHARED))

import torch
from training import set_seed, train_experiment, train_frozen_with_feature_cache
from vmamba_loader import build_vmamba

In [ ]:
set_seed(42)
model, last_stage = build_vmamba()  # genuine architecture, random weights
for parameter in model.parameters():
    parameter.requires_grad = True
total = sum(parameter.numel() for parameter in model.parameters())
trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
assert total == trainable
print(f'Parameters: {total:,}; all {trainable:,} are trainable')

In [ ]:
train_experiment(
    model=model,
    phase='from_scratch',
    last_stage=last_stage,
    output_dir=MODEL_FOLDER / 'results_from_scratch',
    epochs=20,
    learning_rate=1e-4,
    batch_size=2,
    center_crop=False,
)

The saved score is validation accuracy, not independent test accuracy. VMamba-T is large for 176 training images, so substantial overfitting is possible.

## Fine-tuned VMamba — classifier only
Start from the best-validation-loss checkpoint of the 20-epoch scratch run above. Train only its existing classifier for 20 additional epochs and verify that every body tensor stays identical. This stage depends on the scratch training; separate epoch numbering does not make it an independent experiment.

In [ ]:
set_seed(42)
fine_tuned_model, _ = build_vmamba()
checkpoint = MODEL_FOLDER / 'results_from_scratch' / 'best_validation_loss.pth'
fine_tuned_model.load_state_dict(torch.load(checkpoint, map_location='cpu', weights_only=True))
body_before = {k: v.detach().cpu().clone() for k, v in fine_tuned_model.backbone.state_dict().items()}
train_frozen_with_feature_cache(
    model=fine_tuned_model,
    output_dir=MODEL_FOLDER / 'results_fine_tuned',
    epochs=20, learning_rate=1e-4,
    image_batch_size=2, classifier_batch_size=16, center_crop=False,
)
assert all(torch.equal(body_before[k], v.detach().cpu()) for k, v in fine_tuned_model.backbone.state_dict().items()), 'Body changed during classifier training'
print('Verified: every VMamba body tensor is unchanged.')